# Nettoyage des courriels des Decrypteurs

### importation des modules custom

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..')) +'\\analyser_tools'
print(module_path)
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from EmailDataFrame import *

### import python modules

In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import logging
%matplotlib inline

### definir les paths vers les donn?es

In [ ]:
cwd = pathlib.PurePath(os.getcwd())
project_root_dir = cwd.parent
theme_path = project_root_dir / "data" / "theme_words.csv"
email_raw_path = project_root_dir / "output" / 'csv_file' / 'email_raw.csv'
facebook_url_info_path = project_root_dir / 'output' / 'csv_file' / 'facebook_urls_info.csv'
youtube_url_info_path = project_root_dir / 'output' / 'csv_file' / 'youtube_urls_info.csv'
log_file_path = project_root_dir / 'output' / 'log_file' / 'url_cleaning.log'
removed_email_path = project_root_dir / 'output' / 'csv_file' / 'removed_email_from_decrypteur.csv'
dataclean_pickle_path = project_root_dir / 'output' / 'pickle_obj' / 'dataClean'


### D?finir le logging

In [ ]:
logging.basicConfig(filename=log_file_path, format="%(message)s", filemode='w', level=logging.INFO)


### load data et nettoyage

In [ ]:
email_raw_converters = {"attach_type": lambda x: x.strip("[]").replace("'", "").lower().strip().split(", "),
                            "urls": lambda x: x.strip("[]").replace("'", "").strip().split(', ')
                            }
email_df_raw = EmailDF.from_csv(csv_path=email_raw_path, converters=email_raw_converters)
email_df_raw.read_dict_theme_from_csv(theme_path)

debut = (2020, 3, 1) # 1er mars 2020 pour le premier courriel
fin = (9999, 12, 30) # infini

email_df = (email_df_raw.filt_by_date(start_date=debut, end_date=fin) #conserver entre debut et fin
            .remove_email_by_from(logpath=removed_email_path) #log les courriels retirer 
            .clean_urls() #courriel spam (antivirus), repetition de domaine
            .add_domain_column() #trouver domaine des urls restants
            .add_has_url_column() #ajouter s'il y a des urls ou non
            .add_text_column() # combiner body et titre du courriel, puis retirer emoji, non-roman text, saut de ligne et tab
            .add_theme_column() # tagger les courriels ? partir du text
            .add_source_column()) # trouver les sources (comme domaine, mais sans r?p?tition)

### Des courriels en datant de :

In [ ]:
print("DE ",email_df_raw.df.datetime.min()," ? ", email_df_raw.df.datetime.max())

### Nombre de courriels

In [ ]:
email_df_raw.df.shape[0]

### Nombre apr?s nettoyage

In [ ]:
email_df.df.shape[0]

### courbe avant filtre par date

In [ ]:
email_df_raw.df.set_index("datetime").resample('D').count().email_id.plot()

### Apr?s filtre par date

In [ ]:
email_df.df.set_index("datetime").resample('D').count().email_id.plot()

### Courriels avant 1er mars retir?

In [ ]:
email_df_raw.df.shape[0] - email_df_raw.filt_by_date(debut, fin).df.shape[0]

### Courriels de r?ponse des d?crypteurs retir?

In [ ]:
email_df_raw.filt_by_date(debut, fin).df.shape[0] - email_df_raw.filt_by_date(debut, fin).remove_email_by_from().df.shape[0]

### Restant ? analyser

In [ ]:
email_df_raw.filt_by_date(debut, fin).remove_email_by_from().df.shape[0]

### Nettoyage des URLs

In [ ]:
# df_log = pd.read_clipboard(header=None) #create csv from log file using excel and copy everything before running this cell


#### Nombre d'URLs modifi? ou supprim?

In [ ]:
df_log.groupby(0).count()[1]

### exemple modif

### Youtube

In [ ]:
df_log[df_log[0]=='yt_url_modif'][[0,1,2]]

### Facebook

In [ ]:
df_log[df_log[0]=='fb_url_modif'][[0,1,2]]

### Exemple supprimer pour spam (domain in ['microsoft', 'can01', 'safelinks', 'mailtrack', 'avast', 'aka', 'avg', 'symantec', 'xx']

In [ ]:
df_log[df_log[0]=='spam_url_log'][[0,1,2]]

### retir? car il y avait deja 5 urls du meme domaine dans le courriel

In [ ]:
df_log[df_log[0]=='rep_url_removed'][[0,1,2]]

### retrait par courriel en raison d'une r?p?tion de domaine dans un m?me courriel

In [ ]:
df_log[df_log[0]=='rep_url_removed'][[0,1,2]][1].value_counts()